[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bulentsoykan/ABM-particle-filter-notebooks/blob/main/notebook_05_scaling.ipynb)

# Module 5: The Curse of Dimensionality

**Learning objectives**
1. Understand why the number of required particles grows exponentially with agents
2. Visualize the collision-count explosion that drives complexity
3. Reproduce the paper's error heatmap 
4. Understand the fundamental trade-off: model fidelity vs computational cost



---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color':      '#8b949e', 'ytick.color':     '#8b949e',
    'text.color':       '#e6edf3', 'grid.color':      '#21262d',
    'grid.alpha': 0.5,  'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'axes.grid': True,
    'font.size': 11,
})
os.makedirs('figures', exist_ok=True)
print("Libraries loaded.")

In [ ]:
# ── Minimal StationSim-style corridor ABM ──────────────────────────────────────

class Agent:
    '''A pedestrian moving left to right through a corridor.

    The ONLY source of randomness: when a faster agent catches up to a slower one,
    it randomly dodges LEFT or RIGHT (50/50 coin flip).  Everything else is
    deterministic.  That one binary choice is all it takes to cause divergence.
    '''
    def __init__(self, agent_id, env_width=100, env_height=50, rng=None):
        rng = rng or np.random
        self.id         = agent_id
        self.x          = rng.uniform(0, 3)
        self.y          = rng.uniform(5, env_height - 5)
        self.max_speed  = max(0.3, rng.normal(1.0, 0.4))
        self.env_width  = env_width
        self.env_height = env_height
        self.active     = True
        self.traj_x     = [self.x]
        self.traj_y     = [self.y]

    def step(self, all_agents, separation=4.5, rng=None):
        rng = rng or np.random
        if not self.active:
            self.traj_x.append(self.x)
            self.traj_y.append(self.y)
            return
        speed, dy = self.max_speed, 0.0
        for other in all_agents:
            if other.id == self.id or not other.active:
                continue
            dx   = other.x - self.x
            dist = np.hypot(dx, other.y - self.y)
            if 0 < dx < separation * 2 and dist < separation:
                speed *= 0.6
                dy = 1.0 if rng.random() < 0.5 else -1.0   # THE random choice
                break
        self.x = min(self.x + speed, self.env_width)
        self.y = np.clip(self.y + dy, 2, self.env_height - 2)
        if self.x >= self.env_width:
            self.active = False
        self.traj_x.append(self.x)
        self.traj_y.append(self.y)


class CorridorABM:
    '''Corridor where N agents walk left-to-right.

    State vector: [x0,y0, x1,y1, ..., x_{N-1},y_{N-1}]  — dimension 2*N_agents.
    '''
    def __init__(self, n_agents=10, env_width=100, env_height=50, seed=None):
        rng             = np.random.RandomState(seed)
        self.n_agents   = n_agents
        self.env_width  = env_width
        self.env_height = env_height
        self.rng        = rng
        self.agents     = [Agent(i, env_width, env_height, rng)
                           for i in range(n_agents)]
        self.pos_history = [self.get_positions()]
        self.step_count  = 0
        self.collisions  = 0

    def get_positions(self):
        return np.array([[a.x, a.y] for a in self.agents])

    def get_state_vector(self):
        '''Flat [x0,y0, x1,y1,...] — used by the particle filter.'''
        return self.get_positions().flatten()

    def step(self):
        before = [a.active for a in self.agents]
        for agent in self.agents:
            agent.step(self.agents, rng=self.rng)
        # count collision events (agents that slowed down)
        for a in self.agents:
            for b in self.agents:
                if a.id >= b.id or not a.active or not b.active:
                    continue
                if np.hypot(a.x - b.x, a.y - b.y) < 4.5:
                    self.collisions += 1
        self.pos_history.append(self.get_positions())
        self.step_count += 1

    def run(self, n_steps):
        for _ in range(n_steps):
            self.step()
        return self

print("CorridorABM defined.  State dimension = 2 x N_agents.")


In [ ]:
def systematic_resample(weights):
    '''Systematic resampling — O(N), low variance, preferred for PF.

    Algorithm :
      1. Draw one random U ~ Uniform[0, 1/N]
      2. Space N points evenly: U, U+1/N, U+2/N, ..., U+(N-1)/N
      3. Walk the CDF; each particle covers a slice proportional to its weight
         -> heavy particles get sampled multiple times, light ones get dropped
    '''
    N   = len(weights)
    w   = np.asarray(weights, dtype=float)
    w   = w / w.sum()
    U   = np.random.uniform(0, 1.0 / N)
    pts = (np.arange(N) + U) / N
    cdf = np.cumsum(w)
    i, j, idxs = 0, 0, np.zeros(N, dtype=int)
    while i < N:
        if pts[i] <= cdf[j]:
            idxs[i] = j
            i += 1
        else:
            j = min(j + 1, N - 1)
    return idxs

from copy import deepcopy

class ParticleFilterABM:
    '''SIR Particle Filter applied to CorridorABM.

    Each 'particle' is a full ABM instance.  Every resample_window steps:
      PREDICT  -> advance all N_p ABMs one step
      REWEIGHT -> score each particle against the pseudo-truth observation
      RESAMPLE -> duplicate high-weight particles, discard low-weight ones
      JITTER   -> add small Gaussian noise to prevent particle deprivation

    '''
    def __init__(self, n_particles, n_agents, env_width=100, env_height=50,
                 particle_std=0.25, resample_window=20):
        self.n_p    = n_particles
        self.n_a    = n_agents
        self.pstd   = particle_std
        self.window = resample_window
        self.particles = [
            CorridorABM(n_agents, env_width, env_height, seed=i)
            for i in range(n_particles)
        ]
        self.weights     = np.ones(n_particles) / n_particles
        self.w_history   = []    # weights at each update (for degeneracy viz)
        self.err_history = []    # mean weighted error at each update
        self.step_count  = 0

    def predict(self):
        for p in self.particles:
            p.step()
        self.step_count += 1

    def update(self, observation):
        '''Reweight + resample against observed state vector.'''
        # Eq.(3): epsilon_i = ||y_k - x_k^i||_2
        errors = np.array([
            np.linalg.norm(p.get_state_vector() - observation)
            for p in self.particles
        ])
        # Weight: w_i = eta / (1e-9 + epsilon_i)
        self.weights = 1.0 / (1e-9 + errors)
        self.weights /= self.weights.sum()
        self.w_history.append(self.weights.copy())
        self.err_history.append(float(np.dot(self.weights, errors)))

        # Systematic resample
        idxs = systematic_resample(self.weights)
        self.particles = [deepcopy(self.particles[k]) for k in idxs]
        self.weights   = np.ones(self.n_p) / self.n_p

        # Jitter / roughening: prevents all particles collapsing to one state
        if self.pstd > 0:
            for p in self.particles:
                noise = np.random.randn(self.n_a, 2) * self.pstd
                for k, ag in enumerate(p.agents):
                    ag.x = np.clip(ag.x + noise[k, 0], 0, p.env_width)
                    ag.y = np.clip(ag.y + noise[k, 1], 2, p.env_height - 2)

    def get_estimate(self):
        '''Weighted mean state estimate (Eq. 7).'''
        states = np.array([p.get_state_vector() for p in self.particles])
        return np.average(states, weights=self.weights, axis=0)

    def best_particle(self):
        return self.particles[np.argmax(self.weights)]

    def run(self, truth_model, n_steps):
        '''Run PF alongside the truth model for n_steps.'''
        for t in range(n_steps):
            self.predict()
            truth_model.step()
            if (t + 1) % self.window == 0:
                self.update(truth_model.get_state_vector())
        return self

print("ParticleFilterABM defined.")


## Part 1: Why More Agents = Exponential Complexity

Each near-collision between two agents forces a random binary choice: left or right.

- With **N** agents there are up to **N*(N-1)/2** possible collision pairs per step
- The particle filter needs at least one particle that gets *every* random choice right
- The probability of that happening falls exponentially as N grows

Let's measure the actual collision count empirically.


In [ ]:
# Measure collision counts across different agent counts
np.random.seed(42)
agent_counts  = list(range(2, 42, 2))
n_runs        = 8
n_steps_coll  = 80
collision_data = {}

for n_a in agent_counts:
    counts = []
    for run in range(n_runs):
        m = CorridorABM(n_agents=n_a, seed=run * 100)
        m.run(n_steps_coll)
        counts.append(m.collisions)
    collision_data[n_a] = counts
    print(f"  N_agents = {n_a:2d}  |  mean collisions: {np.mean(counts):6.0f}")

print("\nCollisions grow super-linearly (roughly O(N^2)) with N_agents.")


In [ ]:
from scipy.stats import linregress

na_arr   = np.array(agent_counts)
mean_col = np.array([np.mean(collision_data[n]) for n in agent_counts])
std_col  = np.array([np.std(collision_data[n])  for n in agent_counts])

# Fit polynomial (degree 2) in log-log space for reference
log_na  = np.log(na_arr)
log_col = np.log(mean_col + 1)
slope, intercept, r, *_ = linregress(log_na, log_col)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Collision Growth — The Source of Exponential Complexity',
             fontsize=14, fontweight='bold')

# Linear scale
ax1.fill_between(na_arr, mean_col - std_col, mean_col + std_col, alpha=0.25, color='#58a6ff')
ax1.plot(na_arr, mean_col, color='#58a6ff', lw=2.5, marker='o', ms=6, label='Mean collisions')
ax1.set(xlabel='Number of agents', ylabel='Total collisions in 80 steps',
        title='Linear Scale')
ax1.legend(fontsize=9)

# Log-log scale with fit
fit_line = np.exp(intercept) * na_arr ** slope
ax2.scatter(na_arr, mean_col, color='#58a6ff', s=50, zorder=4, label='Measured')
ax2.plot(na_arr, fit_line, color='#f0883e', lw=2, ls='--',
         label=f'Power law fit: O(N^{slope:.2f})')
ax2.set(xscale='log', yscale='log',
        xlabel='Number of agents (log)', ylabel='Collisions (log)',
        title=f'Log-Log Scale  (slope = {slope:.2f})')
ax2.legend(fontsize=9)
ax2.text(0.05, 0.05,
         f'Each extra agent adds\n~{slope:.1f}x more interactions\n=> more stochasticity\n=> more particles needed',
         transform=ax2.transAxes, va='bottom', fontsize=9,
         bbox=dict(boxstyle='round', fc='#0d1117', ec='#58a6ff', alpha=0.9))

plt.tight_layout()
plt.savefig('figures/fig_05a_collision_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Error Heatmap

We now sweep over (N_agents, N_particles) combinations and measure the median PF error.

**Reading the heatmap:**
- Dark = low error (filter working well)
- Bright = high error (filter failing)
- The "staircase" shows how many particles you need to keep error low for each agent count


In [ ]:
# Build the error heatmap (5x5 grid — runs in ~2 min)
# To reproduce the full paper Fig 5, increase the ranges below.

n_runs_hm   = 3
agent_range = [2, 5, 10, 15, 20]
part_range  = [5, 10, 25, 50, 100]
n_steps_hm  = 40
window_hm   = 20

heatmap = np.zeros((len(part_range), len(agent_range)))

total = len(agent_range) * len(part_range)
done  = 0
for j, n_a in enumerate(agent_range):
    for i, n_p in enumerate(part_range):
        run_errs = []
        for run in range(n_runs_hm):
            truth_hm = CorridorABM(n_agents=n_a, seed=run)
            pf_hm    = ParticleFilterABM(n_particles=n_p, n_agents=n_a,
                                          particle_std=0.25,
                                          resample_window=window_hm)
            pf_hm.run(truth_hm, n_steps_hm)
            if pf_hm.err_history:
                run_errs.append(np.mean(pf_hm.err_history))
        heatmap[i, j] = np.median(run_errs) if run_errs else np.nan
        done += 1
        print(f"  [{done:3d}/{total}]  N_a={n_a:2d}, N_p={n_p:4d}  "
              f"-> median error = {heatmap[i,j]:.2f}", end='\r')

print(f"\nHeatmap complete. Shape: {heatmap.shape}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Error Heatmap (Figure 5 reproduction) — Malleson et al. 2020',
             fontsize=14, fontweight='bold')

# ── Linear particle axis ──────────────────────────────────────────────────────
im1 = axes[0].imshow(heatmap, aspect='auto', origin='lower',
                     cmap='plasma_r', interpolation='bilinear',
                     vmin=0, vmax=np.nanpercentile(heatmap, 90))
axes[0].set(xticks=range(len(agent_range)), xticklabels=agent_range,
            yticks=range(len(part_range)), yticklabels=part_range,
            xlabel='Number of agents (N_a)',
            ylabel='Number of particles (N_p)',
            title='Linear particle axis')
plt.colorbar(im1, ax=axes[0], label='Median mean error (m)')

# ── Log particle axis ─────────────────────────────────────────────────────────
im2 = axes[1].imshow(heatmap, aspect='auto', origin='lower',
                     cmap='plasma_r', interpolation='bilinear',
                     vmin=0, vmax=np.nanpercentile(heatmap, 90))
axes[1].set(xticks=range(len(agent_range)), xticklabels=agent_range,
            yticks=range(len(part_range)),
            yticklabels=[f'{p} (log={np.log(p):.1f})' for p in part_range],
            xlabel='Number of agents (N_a)',
            ylabel='log(N_p)',
            title='Log particle axis (matches paper Fig. 5)')
plt.colorbar(im2, ax=axes[1], label='Median mean error (m)')

# Annotate the "good region" boundary
for ax in axes:
    ax.text(0.02, 0.97,
            'Dark = low error\nBright = filter failing\n\nNeed exponentially more\nparticles per extra agent',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', fc='#0d111799', ec='white', alpha=0.9))

plt.tight_layout()
plt.savefig('figures/fig_05b_error_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 3: PF vs Kalman Filter — Why Not Just Use KF?

The **Kalman Filter** (KF) is the classical data assimilation method.
It's computationally much cheaper — O(N^3) in state dimension.

| Method | Complexity | Works when... |
|--------|-----------|---------------|
| Kalman Filter | O(n^3) per step | Model is **linear** and noise is **Gaussian** |
| Extended KF | O(n^3) | Model is **smooth nonlinear** |
| Particle Filter | O(N_p * n) | Works for **any** model, including agent-based |

An ABM violates both KF assumptions:
1. **Non-linear**: agent interactions (collision, avoidance) are deeply nonlinear
2. **Non-Gaussian**: the dodge-left/dodge-right decision creates multimodal distributions

The PF can handle this — but at the cost of exponential particle scaling.


In [ ]:
# Illustrate the non-Gaussian distribution of agent positions after collisions
np.random.seed(77)
N_SAMPLES = 2000
N_A_NG    = 3
N_S_NG    = 30

# Collect many single-agent positions at step 30
final_x = []
for seed in range(N_SAMPLES):
    m = CorridorABM(n_agents=N_A_NG, seed=seed)
    m.run(N_S_NG)
    final_x.append(m.agents[0].x)
final_x = np.array(final_x)

# KF would assume Gaussian — let's see if that's true
from scipy.stats import norm as norm_dist
mu_kf  = final_x.mean()
sig_kf = final_x.std()
x_plot = np.linspace(final_x.min() - 2, final_x.max() + 2, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Non-Gaussian State Distribution — Why KF Fails for ABMs',
             fontsize=14, fontweight='bold')

axes[0].hist(final_x, bins=60, density=True, color='#58a6ff', alpha=0.7,
             label='Actual distribution')
axes[0].plot(x_plot, norm_dist.pdf(x_plot, mu_kf, sig_kf),
             color='#f85149', lw=2.5, label=f'KF Gaussian fit (mu={mu_kf:.1f}, sig={sig_kf:.1f})')
axes[0].set(xlabel=f'X position of agent 0 at step {N_S_NG}',
            ylabel='Probability density',
            title='Agent position distribution (N=2000 runs)')
axes[0].legend(fontsize=9)
axes[0].text(0.02, 0.97,
             'Non-Gaussian: modes\ncorrespond to different\ncollision histories',
             transform=axes[0].transAxes, va='top', fontsize=9,
             bbox=dict(boxstyle='round', fc='#0d1117', ec='#58a6ff', alpha=0.9))

# KF vs PF error comparison (simulated)
n_a_range = np.array([2, 4, 6, 8, 10, 12])
# KF diverges fast with nonlinear ABMs (schematic, based on paper discussion)
kf_error  = 0.5 + 0.8 * n_a_range ** 1.5        # grows polynomially but still fails
pf_error_many = 0.3 + 0.15 * np.exp(0.15 * n_a_range)   # low with enough particles
pf_error_few  = 0.8 + 0.3  * np.exp(0.2  * n_a_range)   # high with few particles

axes[1].plot(n_a_range, kf_error,      color='#f85149', lw=2.5, ls='--',
             marker='s', ms=6, label='Kalman Filter (misspecified)')
axes[1].plot(n_a_range, pf_error_few,  color='#f0883e', lw=2.5, ls=':',
             marker='^', ms=6, label='PF (few particles)')
axes[1].plot(n_a_range, pf_error_many, color='#3fb950', lw=2.5,
             marker='o', ms=6, label='PF (enough particles)')
axes[1].set(xlabel='Number of agents', ylabel='Tracking error (m)',
            title='Schematic: KF vs PF Error Scaling')
axes[1].legend(fontsize=9)
axes[1].text(0.02, 0.97,
             'KF: wrong model assumptions\n=> systematic error\n\nPF: right tool, but\nneeds many particles',
             transform=axes[1].transAxes, va='top', fontsize=9,
             bbox=dict(boxstyle='round', fc='#0d1117', ec='#3fb950', alpha=0.9))

plt.tight_layout()
plt.savefig('figures/fig_05c_kf_vs_pf.png', dpi=150, bbox_inches='tight')
plt.show()

## Final Summary — The Big Picture

```
AGENT-BASED MODEL                PARTICLE FILTER
─────────────────                ───────────────────────────────────
Captures emergence               Handles non-linear, non-Gaussian models
Models heterogeneity             Works with any state representation
Stochastic (diverges)            Corrects drift using real-time data
Cannot be updated mid-run        Each particle is a full ABM copy

THE COMBINATION:
  N_p copies of the ABM run in parallel
  Each window, they are scored against observations
  Good particles survive, bad ones are replaced
  -> Real-time crowd state estimation

THE CHALLENGE:
  N_p must grow exponentially with N_agents
  For 40 agents -> ~10,000 particles needed (paper, Fig 5)
  For 1000 agents -> beyond current compute
  -> Active research area (GPU PF, spatial decomposition, ...)
```

